# 🚗 Autonomous Vehicle — Behavioral Cloning

**Project:** Supervised learning model to predict steering angles from dashcam images.

**Dataset:** Udacity Self-Driving Car Dataset (images + `driving_log.csv`)

**Architecture:** NVIDIA End-to-End CNN (built from scratch)

---
## Sections
1. Install & Import Libraries
2. Load and Explore the Dataset
3. Data Augmentation
4. Data Generator
5. Build the Model (NVIDIA CNN)
6. Train the Model
7. Evaluate & Visualize Results
8. Make Predictions on Test Images

## 1. Install & Import Libraries

In [ ]:
# Install required libraries (run once)
# !pip install tensorflow opencv-python matplotlib pandas scikit-learn numpy

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, Flatten, Dense, Dropout, Lambda, Cropping2D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

print('TensorFlow version:', tf.__version__)
print('All libraries imported successfully!')

## 2. Load and Explore the Dataset

> **Setup Instructions:**
> 1. Download the Udacity dataset from: https://d17h27t6h515a5.cloudfront.net/topher/2016/December/584f6edd_data/data.zip
> 2. Extract the zip file into the project folder so the structure looks like:
> ```
> autuonmus vehicle/
>     data/
>         driving_log.csv
>         IMG/
>             center_*.jpg
>             left_*.jpg
>             right_*.jpg
>     Behavioral_Cloning.ipynb
> ```

In [ ]:
# ─── CONFIGURATION ───────────────────────────────────────────────────────────
DATA_PATH   = './data/'          # Folder containing driving_log.csv and IMG/
LOG_FILE    = os.path.join(DATA_PATH, 'driving_log.csv')
IMG_HEIGHT  = 66                 # NVIDIA CNN input height
IMG_WIDTH   = 200                # NVIDIA CNN input width
IMG_CHANNELS = 3
CORRECTION  = 0.2               # Steering correction for left/right cameras
BATCH_SIZE  = 32
EPOCHS      = 10
LEARNING_RATE = 1e-4
VALIDATION_SPLIT = 0.2
# ─────────────────────────────────────────────────────────────────────────────

# Load the CSV log file
columns = ['center', 'left', 'right', 'steering', 'throttle', 'brake', 'speed']
df = pd.read_csv(LOG_FILE, names=columns, header=None)
df.head()

In [ ]:
print(f'Total samples in driving log: {len(df)}')
print('\nBasic Stats for Steering Angle:')
print(df['steering'].describe())

In [ ]:
# Visualize the distribution of steering angles
plt.figure(figsize=(10, 4))
plt.hist(df['steering'], bins=50, color='steelblue', edgecolor='black')
plt.title('Distribution of Steering Angles in Dataset')
plt.xlabel('Steering Angle')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()
print('Notice the large spike at 0 (straight driving). Augmentation will help balance this.')

In [ ]:
# Build a flat list of (image_path, steering_angle) using all 3 cameras
samples = []

for _, row in df.iterrows():
    steering_center = float(row['steering'])
    steering_left   = steering_center + CORRECTION   # left camera: steer right
    steering_right  = steering_center - CORRECTION   # right camera: steer left

    # Extract just the filename and build the path
    for camera, angle in [('center', steering_center),
                           ('left',   steering_left),
                           ('right',  steering_right)]:
        img_filename = row[camera].strip().split('/')[-1]
        img_path = os.path.join(DATA_PATH, 'IMG', img_filename)
        samples.append((img_path, angle))

print(f'Total samples (center + left + right cameras): {len(samples)}')

# Split into training and validation sets
train_samples, val_samples = train_test_split(samples, test_size=VALIDATION_SPLIT, random_state=42)
print(f'Training samples  : {len(train_samples)}')
print(f'Validation samples: {len(val_samples)}')

In [ ]:
# Show a sample of the raw images
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cameras = ['center', 'left', 'right']
for i, cam in enumerate(cameras):
    img_filename = df[cam].iloc[5].strip().split('/')[-1]
    img_path = os.path.join(DATA_PATH, 'IMG', img_filename)
    img = mpimg.imread(img_path)
    axes[i].imshow(img)
    axes[i].set_title(f'{cam.capitalize()} Camera')
    axes[i].axis('off')
plt.suptitle('Sample Raw Images from Dataset', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Data Augmentation

To prevent overfitting and make the model robust, we apply these augmentations:
- **Random Flip:** Mirror the image horizontally and negate the steering angle.
- **Random Pan (Translation):** Shift the image horizontally/vertically to simulate lane offsets.
- **Random Brightness:** Randomly darken or brighten the image to simulate different lighting.
- **Preprocessing:** Crop sky/hood, resize to 66x200, convert to YUV colorspace (as per NVIDIA paper).

In [ ]:
def preprocess_image(img):
    """
    Preprocess a single image:
    1. Crop top 60px (sky) and bottom 25px (car hood)
    2. Resize to NVIDIA input: 66 x 200
    3. Convert BGR -> YUV colorspace
    """
    img = img[60:-25, :, :]                        # Crop
    img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))  # Resize
    img = cv2.cvtColor(img, cv2.COLOR_BGR2YUV)      # YUV
    return img


def random_flip(img, angle):
    """Randomly flip the image horizontally and negate the steering angle."""
    if random.random() > 0.5:
        img   = cv2.flip(img, 1)
        angle = -angle
    return img, angle


def random_pan(img, angle):
    """Randomly pan (translate) the image to simulate lane shift."""
    pan_x = 100 * (random.random() - 0.5)   # shift ±50 pixels horizontally
    pan_y = 10  * (random.random() - 0.5)   # shift ±5  pixels vertically
    M = np.float32([[1, 0, pan_x], [0, 1, pan_y]])
    height, width = img.shape[:2]
    img = cv2.warpAffine(img, M, (width, height))
    angle += pan_x * 0.002               # adjust steering for horizontal pan
    return img, angle


def random_brightness(img):
    """Randomly adjust brightness to simulate day/night/shadow conditions."""
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    factor  = 0.25 + random.random() * 1.0    # brightness scale factor
    img_hsv[:, :, 2] = np.clip(img_hsv[:, :, 2].astype(np.float32) * factor, 0, 255)
    img = cv2.cvtColor(img_hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    return img


def augment(img, angle):
    """Apply all augmentations to an image and steering angle."""
    img, angle = random_pan(img, angle)
    img        = random_brightness(img)
    img, angle = random_flip(img, angle)
    return img, angle


print('Augmentation functions defined.')

In [ ]:
# Visualize how augmentation changes an image
sample_path, sample_angle = train_samples[0]
sample_img = cv2.imread(sample_path)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

# Original
axes[0].imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original')
axes[0].axis('off')

# Apply augmentations one by one
aug_labels = ['Augmented #1', 'Augmented #2', 'Augmented #3', 'Preprocessed']
for i, label in enumerate(aug_labels[:-1]):
    aug_img, aug_angle = augment(sample_img.copy(), sample_angle)
    axes[i+1].imshow(cv2.cvtColor(aug_img, cv2.COLOR_BGR2RGB))
    axes[i+1].set_title(f'{label}\nAngle: {aug_angle:.3f}')
    axes[i+1].axis('off')

# Preprocessed
pre_img = preprocess_image(sample_img.copy())
axes[4].imshow(pre_img)  # YUV - colors look different, that's expected
axes[4].set_title(f'Preprocessed (YUV)\n{IMG_HEIGHT}x{IMG_WIDTH}')
axes[4].axis('off')

plt.suptitle('Data Augmentation Visualization', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Data Generator

The dataset has thousands of images. Loading all of them into RAM at once would cause memory issues. We use a **Keras generator** that loads and processes images in small batches on-the-fly.

In [ ]:
def data_generator(samples, batch_size=32, is_training=True):
    """
    Keras data generator — yields batches of (images, steering_angles).

    Args:
        samples     : List of (image_path, steering_angle) tuples.
        batch_size  : Number of samples per batch.
        is_training : If True, apply augmentation. If False (validation), only preprocess.
    """
    num_samples = len(samples)

    while True:   # Loop forever — Keras will call this generator endlessly
        samples = shuffle(samples)

        for offset in range(0, num_samples, batch_size):
            batch = samples[offset : offset + batch_size]

            images = []
            angles = []

            for img_path, angle in batch:
                img = cv2.imread(img_path)
                if img is None:
                    continue  # skip missing images

                if is_training:
                    img, angle = augment(img, angle)

                img = preprocess_image(img)
                img = img.astype(np.float32) / 255.0   # Normalize to [0, 1]

                images.append(img)
                angles.append(angle)

            yield np.array(images), np.array(angles)


# Create training and validation generators
train_generator = data_generator(train_samples, batch_size=BATCH_SIZE, is_training=True)
val_generator   = data_generator(val_samples,   batch_size=BATCH_SIZE, is_training=False)

print('Data generators ready.')

## 5. Build the Model (NVIDIA End-to-End CNN)

We implement the NVIDIA CNN architecture described in the paper:
*"End to End Learning for Self-Driving Cars"* (Bojarski et al., 2016).

```
Input (66×200×3 YUV image)
  → Conv2D 24 filters, 5×5, stride 2, ELU
  → Conv2D 36 filters, 5×5, stride 2, ELU
  → Conv2D 48 filters, 5×5, stride 2, ELU
  → Conv2D 64 filters, 3×3, ELU
  → Conv2D 64 filters, 3×3, ELU
  → Dropout(0.5)
  → Flatten
  → Dense(100), ELU
  → Dense(50),  ELU
  → Dense(10),  ELU
  → Dense(1)    ← predicted steering angle
```

In [ ]:
def build_nvidia_model(input_shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)):
    """
    Build and return the NVIDIA End-to-End self-driving CNN from scratch.
    Reference: https://arxiv.org/abs/1604.07316
    """
    model = Sequential([
        # ── Convolutional Layers ──────────────────────────────────
        Conv2D(24, (5, 5), strides=(2, 2), activation='elu',
               input_shape=input_shape, name='conv1'),
        Conv2D(36, (5, 5), strides=(2, 2), activation='elu', name='conv2'),
        Conv2D(48, (5, 5), strides=(2, 2), activation='elu', name='conv3'),
        Conv2D(64, (3, 3), activation='elu', name='conv4'),
        Conv2D(64, (3, 3), activation='elu', name='conv5'),

        # ── Regularization ───────────────────────────────────────
        Dropout(0.5),

        # ── Fully Connected Layers ───────────────────────────────
        Flatten(),
        Dense(100, activation='elu', name='fc1'),
        Dense(50,  activation='elu', name='fc2'),
        Dense(10,  activation='elu', name='fc3'),

        # ── Output: single steering angle ────────────────────────
        Dense(1, name='output')   # Linear activation (regression)
    ])

    return model


# Build and display the model summary
model = build_nvidia_model()
model.summary()

## 6. Train the Model

In [ ]:
# Compile the model
# Loss: Mean Squared Error (regression task — minimize difference between predicted and true angle)
# Optimizer: Adam (adaptive learning rate)
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='mse',
    metrics=['mae']   # Mean Absolute Error as an extra metric
)

print(f'Model compiled.')
print(f'  Optimizer     : Adam (lr={LEARNING_RATE})')
print(f'  Loss function : Mean Squared Error')
print(f'  Extra metric  : Mean Absolute Error')

In [ ]:
# ── Callbacks ────────────────────────────────────────────────────────────────
checkpoint = ModelCheckpoint(
    'best_model.h5',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,        # stop if val_loss doesn't improve for 3 epochs
    restore_best_weights=True,
    verbose=1
)

# ── Train ─────────────────────────────────────────────────────────────────────
steps_per_epoch   = len(train_samples) // BATCH_SIZE
validation_steps  = len(val_samples)   // BATCH_SIZE

print(f'Starting training...')
print(f'  Epochs           : {EPOCHS}')
print(f'  Steps per epoch  : {steps_per_epoch}')
print(f'  Validation steps : {validation_steps}')
print()

history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_generator,
    validation_steps=validation_steps,
    epochs=EPOCHS,
    callbacks=[checkpoint, early_stop],
    verbose=1
)

print('\nTraining complete!')

## 7. Evaluate & Visualize Results

In [ ]:
# Plot Training vs. Validation Loss
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MSE Loss
axes[0].plot(history.history['loss'],     label='Training Loss',   color='steelblue')
axes[0].plot(history.history['val_loss'], label='Validation Loss', color='coral')
axes[0].set_title('Model Loss (MSE) Over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Mean Squared Error')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history['mae'],     label='Training MAE',   color='steelblue')
axes[1].plot(history.history['val_mae'], label='Validation MAE', color='coral')
axes[1].set_title('Mean Absolute Error Over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean Absolute Error (radians)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_results.png', dpi=120)
plt.show()
print('Plot saved as training_results.png')

## 8. Make Predictions on Test Images

In [ ]:
# Load the best saved model and test on a few validation images
from tensorflow.keras.models import load_model

best_model = load_model('best_model.h5')
print('Best model loaded from best_model.h5')

# Pick 6 random samples from validation set
test_samples = random.sample(val_samples, min(6, len(val_samples)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (img_path, true_angle) in enumerate(test_samples):
    # Load and preprocess
    img = cv2.imread(img_path)
    processed = preprocess_image(img)
    normalized = processed.astype(np.float32) / 255.0
    
    # Predict
    input_tensor = np.expand_dims(normalized, axis=0)   # shape: (1, 66, 200, 3)
    predicted_angle = best_model.predict(input_tensor, verbose=0)[0][0]

    # Display
    axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    error = abs(predicted_angle - true_angle)
    color = 'green' if error < 0.05 else 'orange' if error < 0.15 else 'red'
    axes[i].set_title(
        f'True: {true_angle:.4f}  |  Predicted: {predicted_angle:.4f}\n'
        f'Error: {error:.4f}',
        color=color
    )
    axes[i].axis('off')

plt.suptitle('Steering Angle Predictions vs Ground Truth', fontsize=15)
plt.tight_layout()
plt.savefig('predictions.png', dpi=120)
plt.show()
print('Predictions saved as predictions.png')

In [ ]:
# Final evaluation on the entire validation set
print('Evaluating model on full validation set...')

all_true = []
all_pred = []

for img_path, true_angle in val_samples:
    img = cv2.imread(img_path)
    if img is None:
        continue
    processed  = preprocess_image(img).astype(np.float32) / 255.0
    pred_angle = best_model.predict(np.expand_dims(processed, 0), verbose=0)[0][0]
    all_true.append(true_angle)
    all_pred.append(pred_angle)

all_true = np.array(all_true)
all_pred = np.array(all_pred)

mse = np.mean((all_true - all_pred) ** 2)
mae = np.mean(np.abs(all_true - all_pred))

print(f'\n=== Final Validation Metrics ===')
print(f'  MSE (Mean Squared Error)    : {mse:.6f}')
print(f'  MAE (Mean Absolute Error)   : {mae:.6f} radians')
print(f'  RMSE (Root MSE)             : {np.sqrt(mse):.6f}')

# Scatter plot: True vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(all_true, all_pred, alpha=0.3, s=4, color='steelblue')
plt.plot([-1, 1], [-1, 1], 'r--', label='Perfect Prediction')
plt.xlabel('True Steering Angle')
plt.ylabel('Predicted Steering Angle')
plt.title('True vs Predicted Steering Angles')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('true_vs_predicted.png', dpi=120)
plt.show()
print('Scatter plot saved as true_vs_predicted.png')